# Extract Required Skills

In the previous notebook, you used an LLM to summarize the responsibilities of an AI Engineer.

In this notebook, you will use an LLM to extract the concrete skills that AI engineering job postings ask for, save them to `jobs/3-extracted_skills.jsonl`, and then count which skills appear most often across all postings.

In [ ]:
import json
from pathlib import Path
from dotenv import load_dotenv

import pandas as pd
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
load_dotenv(override=True)

## Load the Classified Jobs

This cell reads the JSONL file created by `4-classify-job-postings.ipynb`.

If the file is missing, run the classification notebook first. The repository also includes example data in the `jobs` folder, so you can continue even if you have not run the previous notebooks on your machine.

In [ ]:
classified_jobs_path = Path("jobs") / "2-classified-jobs.jsonl"

if not classified_jobs_path.exists():
    raise FileNotFoundError(
        f"Classified jobs JSONL file not found: {classified_jobs_path.resolve()}. Run 4-classify-job-postings.ipynb first."
    )

if classified_jobs_path.stat().st_size == 0:
    raise ValueError(
        "The classified jobs JSONL file is empty. Run 4-classify-job-postings.ipynb first."
    )

jobs_df = pd.read_json(classified_jobs_path, lines=True)
print(f"Loaded {len(jobs_df)} job postings")

## Keep Only AI Engineering Roles

We only want to extract skills from postings that the LLM classified as true AI engineering roles in the previous notebook.

This cell filters out everything else, so the skill counts later in the notebook are not skewed by unrelated jobs.

In [ ]:
# Keep only the jobs the LLM classified as true AI engineering roles
jobs_df = jobs_df[jobs_df["is_ai_engineering_role"]]

print(f"Loaded {len(jobs_df)} classified AI engineering job postings")

## Define the Skill Categories

We give the model a small fixed set of categories so the extracted skills are easier to compare and group across postings.

For example, `rag` belongs to `ai-engineering`, `postgres` belongs to `databases`, and `kubernetes` belongs to `dev-ops`.

In [ ]:
skill_categories = [
    "ai-engineering",
    "machine-learning",
    "programming-languages",
    "frontend",
    "backend",
    "databases",
    "cloud",
    "dev-ops",
    "other",
]

## Create the Model Client

We use the OpenAI API to extract skills from each job posting.

In [ ]:
client = OpenAI()

## Define the Extraction Instructions

The instructions tell the model what to extract: concise, normalized, lowercase skill names, and only skills that are clearly important for the role.

We also pass the list of categories into the instructions through an f-string so the model can assign each skill to one of them.

In [ ]:
category_text = ", ".join(skill_categories)

instructions = f"""
You extract required skills from AI engineering job postings.

Return concise normalized skill names like 'python', 'rag', 'langchain', 'aws', or 'docker'.
Return all skill names in lowercase.

Only include skills that are clearly important for the role.

Assign each skill to one of these categories:
{category_text}
"""

print(instructions)

## Define the Output Schema

We ask the model for structured output so the response is easy to parse and store.

The schema requires a `skills` array, where each entry has a `skill` string and a `category` restricted to one of our predefined categories.

Reference: [json-schema.org](https://json-schema.org/)

In [ ]:
skill_schema = {
    "type": "object",
    "properties": {
        "skills": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "skill": {"type": "string"},
                    "category": {"type": "string", "enum": skill_categories},
                },
                "required": ["skill", "category"],
                "additionalProperties": False,
            },
        }
    },
    "required": ["skills"],
    "additionalProperties": False,
}

# Example response that matches this schema:
# {
#     "skills": [
#         {"skill": "rag", "category": "ai-engineering"},
#         {"skill": "postgres", "category": "databases"},
#         {"skill": "kubernetes", "category": "dev-ops"},
#     ]
# }

## Extract Skills for Each Posting

This loop sends one LLM request per job posting.

The instructions and schema stay the same for every request, and only the job description changes, so each posting is judged against the same definition.

In [ ]:
skill_results = []

for i, (_, job) in enumerate(jobs_df.iterrows(), start=1):
    title = job["title"]
    description = job["description"]
    job_url = job["job_url"]

    print(f"Extracting skills for job {i}/{len(jobs_df)}: {title}")

    response = client.responses.create(
        model="gpt-5.4-mini",
        instructions=instructions,
        input=description,
        text={
            "format": {
                "type": "json_schema",
                "name": "ai_engineering_skill_extraction",
                "schema": skill_schema,
                "strict": True,
            },
            "verbosity": "low",
        },
    )

    result = json.loads(response.output_text)

    skill_results.append(
        {
            "title": title,
            "job_url": job_url,
            "extracted_skills": result["skills"],
        }
    )

## Preview the Extracted Skills

Before saving the data, we print the extracted skills for the first ten postings so we can sanity-check what the model returned.

In [ ]:
for job in skill_results[:10]:
    labels = [
        f"{skill['skill']} [{skill['category']}]" for skill in job["extracted_skills"]
    ]
    print(f"{job['title']}: {', '.join(labels)}")

## Save the Extracted Skills

We turn the list of model outputs into a DataFrame and save it as a JSONL file at `jobs/3-extracted_skills.jsonl`.

The next step in this notebook reads from this file, and you can also reuse it later for further analysis.

In [ ]:
skills_jsonl_path = Path("jobs") / "3-extracted_skills.jsonl"

skills_by_job = pd.DataFrame(skill_results)
skills_by_job.to_json(
    skills_jsonl_path, orient="records", lines=True, force_ascii=False
)

print(f"Saved extracted skills to: {skills_jsonl_path.resolve()}")

## Count How Often Each Skill Appears

Now we switch from per-job extraction to a small data analysis step.

We flatten the per-job lists of skills into one row per skill, then count how often each `(category, skill)` pair appears across all postings.

In [ ]:
# Before — one row per job, "extracted_skills" holds a list of dicts:
#   row 0: [{"skill": "python", "category": "programming-languages"},
#           {"skill": "rag",    "category": "ai-engineering"}]
#   row 1: [{"skill": "aws",    "category": "cloud"}]
#
# .explode() turns each list element into its own row (still a dict):
#   row 0: {"skill": "python", "category": "programming-languages"}
#   row 0: {"skill": "rag",    "category": "ai-engineering"}
#   row 1: {"skill": "aws",    "category": "cloud"}
exploded_skills = skills_by_job["extracted_skills"].explode()

# json_normalize() turns those dicts into proper columns:
#      skill    category
#   0  python   programming-languages
#   0  rag      ai-engineering
#   1  aws      cloud
all_skills = pd.json_normalize(exploded_skills)

total_jobs = len(skills_by_job)

# Count how often each (category, skill) pair appears, sorted by count desc
skill_counts = (
    all_skills.value_counts(["category", "skill"]).reset_index(name="count")
)

# Human-friendly "12/42 (28.6%)" string
skill_counts["share_of_jobs"] = skill_counts["count"].apply(
    lambda c: f"{c}/{total_jobs} ({round(c / total_jobs * 100, 1)}%)"
)


## Display the Most Common Skills

Finally, we show the most in-demand skills.

We first look at the top 25 skills overall, and then at the top 10 skills inside each category, so you can see which tools and technologies AI engineering job postings actually ask for today.

In [ ]:
# Top 25 skills across all categories
display(Markdown("### Top 25 skills overall"))
display(skill_counts.head(25)[["category", "skill", "share_of_jobs"]])

# Top 10 skills inside each category
for category in skill_counts["category"].unique():
    top_skills = skill_counts[skill_counts["category"] == category].head(10)[
        ["skill", "share_of_jobs"]
    ]
    display(Markdown(f"### Top 10 in `{category}`"))
    display(top_skills)
